# MatMul-Free LM v4: Proper 370M Model vs Pythia-410M Benchmark

**Goal:** Train a 370M parameter MatMul-Free model matching the paper's configuration and properly benchmark against Pythia-410M baseline.

**Key Changes from v3:**
- Model size: 24 layers, hidden_dim=1024 (~370M params)
- Batch size: 4 (minimum for 4 GPUs)
- Sequence length: 1024 (matching paper)
- Autotuning: Re-enable from original repo
- Parallelization: Use DDP for better efficiency

**Training Plan:**
- Train for 10,000 steps initially
- Compare memory, throughput, and quality vs Pythia-410M

In [1]:
# Cell 1: Environment Setup
import os
import sys
from datetime import datetime

# Set cache to local directory
cache_dir = os.path.join(os.getcwd(), 'hf_cache')
os.makedirs(cache_dir, exist_ok=True)
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HF_DATASETS_CACHE'] = cache_dir

print(f"HuggingFace cache: {cache_dir}")
print(f"Working directory: {os.getcwd()}")
print(f"Python: {sys.version}")
print(f"Time: {datetime.now()}")

HuggingFace cache: /home/ec2-user/SageMaker/matmulMM/hf_cache
Working directory: /home/ec2-user/SageMaker/matmulMM
Python: 3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:46:25) [GCC 14.3.0]
Time: 2025-11-14 14:50:41.993384


In [2]:
# Cell 2: Import Libraries
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
import numpy as np
from pathlib import Path
import json

sys.path.append('.')
from src.models.hgrn_bit.configuration_hgrn_bit import HGRNBitConfig
from src.models.hgrn_bit.modeling_hgrn_bit import HGRNBitForCausalLM

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

/home/ec2-user/anaconda3/envs/matmul-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ec2-user/anaconda3/envs/matmul-env/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
Number of GPUs: 4
  GPU 0: NVIDIA A10G (23.7 GB)
  GPU 1: NVIDIA A10G (23.7 GB)
  GPU 2: NVIDIA A10G (23.7 GB)
  GPU 3: NVIDIA A10G (23.7 GB)


## Model Configuration: 370M Parameters

Matching the paper's 370M model:
- 24 layers (vs our previous 12)
- hidden_dim: 1024
- sequence length: 1024 (vs our previous 128)
- vocab_size: 50,000

In [3]:
# Cell 3: Model Configuration
config = HGRNBitConfig(
    vocab_size=50000,
    hidden_size=1024,
    num_hidden_layers=24,  # Increased from 12 to match paper
    max_position_embeddings=2048,
    num_heads=8,
    expand_ratio=1,
    hidden_ratio=4,
    rms_norm_eps=1e-6,
    use_cache=False,
    attn_mode='fused_recurrent',  # Use fused kernels
)

print("Model Configuration:")
print(f"  Vocab size: {config.vocab_size:,}")
print(f"  Hidden size: {config.hidden_size}")
print(f"  Layers: {config.num_hidden_layers}")
print(f"  Heads: {config.num_heads}")
print(f"  Max sequence: {config.max_position_embeddings}")

# Calculate approximate parameter count
# Rough estimate: vocab_embed + layers * (hidden^2 * constants)
vocab_params = config.vocab_size * config.hidden_size
layer_params = config.num_hidden_layers * (config.hidden_size ** 2) * 12  # Rough multiplier
total_params = vocab_params + layer_params
print(f"\nEstimated parameters: ~{total_params/1e6:.0f}M")

Model Configuration:
  Vocab size: 50,000
  Hidden size: 1024
  Layers: 24
  Heads: 8
  Max sequence: 2048

Estimated parameters: ~353M


In [4]:
# Cell 4: Create Model and Check Actual Size
print("Creating 370M MatMul-Free model...")
model = HGRNBitForCausalLM(config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*60}")
print(f"MODEL SIZE VERIFICATION")
print(f"{'='*60}")
print(f"Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.1f}M)")
print(f"\nTarget: ~370M (paper's model)")
print(f"Baseline: Pythia-410M (405M params)")
print(f"\nParameter ratio: {total_params/405e6:.2f}x Pythia-410M")
print(f"{'='*60}")

del model  # Free memory before training
torch.cuda.empty_cache()

HGRNBitForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Creating 370M MatMul-Free model...

MODEL SIZE VERIFICATION
Total parameters: 410,972,160 (411.0M)
Trainable parameters: 410,972,160 (411.0M)

Target: ~370M (paper's model)
Baseline: Pythia-410M (405M params)

Parameter ratio: 1.01x Pythia-410M


## Dataset Preparation

Using SlimPajama with sequence length 1024 (matching paper)

In [5]:
# Cell 5: Dataset Class
class PajamaDataset(Dataset):
    """SlimPajama dataset loader for .ds files"""
    def __init__(self, data_dir, seq_length=1024, max_files=None):
        self.data_dir = Path(data_dir)
        self.seq_length = seq_length
        self.files = sorted(list(self.data_dir.glob('*.ds')))
        
        if max_files:
            self.files = self.files[:max_files]
        
        print(f"Found {len(self.files)} .ds files")
        
        # Calculate total samples
        self.file_sizes = []
        for f in self.files:
            size = f.stat().st_size // 2  # 2 bytes per token
            num_samples = max(0, (size - seq_length) // seq_length)
            self.file_sizes.append(num_samples)
        
        self.cumulative_sizes = np.cumsum([0] + self.file_sizes)
        self.total_samples = sum(self.file_sizes)
        print(f"Total samples: {self.total_samples:,}")
    
    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # Find which file contains this index
        file_idx = np.searchsorted(self.cumulative_sizes[1:], idx, side='right')
        local_idx = idx - self.cumulative_sizes[file_idx]
        
        # Read from file
        filepath = self.files[file_idx]
        offset = local_idx * self.seq_length * 2  # 2 bytes per token
        
        with open(filepath, 'rb') as f:
            f.seek(offset)
            tokens = np.frombuffer(f.read(self.seq_length * 2), dtype=np.uint16)
        
        if len(tokens) < self.seq_length:
            tokens = np.pad(tokens, (0, self.seq_length - len(tokens)))
        
        return torch.from_numpy(tokens.astype(np.int64))

In [6]:
# Cell 6: Load Dataset
dataset = PajamaDataset(
    data_dir='SlimPajama-6B-nanotron/train',
    seq_length=1024,  # Increased from 128
    max_files=None  # Use all files
)

print(f"\nDataset ready:")
print(f"  Sequence length: {dataset.seq_length}")
print(f"  Total samples: {len(dataset):,}")
print(f"  Total tokens: {len(dataset) * dataset.seq_length:,}")

# Test sample
sample = dataset[0]
print(f"\nSample shape: {sample.shape}")
print(f"Sample dtype: {sample.dtype}")
print(f"Token range: [{sample.min()}, {sample.max()}]")

Found 96 .ds files
Total samples: 786,084

Dataset ready:
  Sequence length: 1024
  Total samples: 786,084
  Total tokens: 804,950,016

Sample shape: torch.Size([1024])
Sample dtype: torch.int64
Token range: [1, 29991]


## Training Setup

Using DataParallel for simplicity (can switch to DDP later for better performance)

In [7]:
# Cell 7: Training Configuration
# Training hyperparameters
BATCH_SIZE_PER_GPU = 4  # Increased from 2 to utilize all 4 GPUs
NUM_GPUS = torch.cuda.device_count()
EFFECTIVE_BATCH_SIZE = BATCH_SIZE_PER_GPU * NUM_GPUS
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_STEPS = 10000
WARMUP_STEPS = 500
GRAD_CLIP = 1.0
LOG_INTERVAL = 100
SAVE_INTERVAL = 1000

print("Training Configuration:")
print(f"  Batch size per GPU: {BATCH_SIZE_PER_GPU}")
print(f"  Number of GPUs: {NUM_GPUS}")
print(f"  Effective batch size: {EFFECTIVE_BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Max steps: {MAX_STEPS:,}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print(f"  Gradient clipping: {GRAD_CLIP}")
print(f"\nSequence length: {dataset.seq_length}")
print(f"Tokens per step: {EFFECTIVE_BATCH_SIZE * dataset.seq_length:,}")
print(f"Total training tokens: {EFFECTIVE_BATCH_SIZE * dataset.seq_length * MAX_STEPS / 1e9:.2f}B")

Training Configuration:
  Batch size per GPU: 4
  Number of GPUs: 4
  Effective batch size: 16
  Learning rate: 0.0003
  Weight decay: 0.1
  Max steps: 10,000
  Warmup steps: 500
  Gradient clipping: 1.0

Sequence length: 1024
Tokens per step: 16,384
Total training tokens: 0.16B


In [8]:
# Cell 8: Initialize Model for Training
print("Initializing model for training...")
model = HGRNBitForCausalLM(config)

# Enable gradient checkpointing for memory efficiency
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()
    print("✓ Gradient checkpointing enabled")

# Move to GPU and parallelize
if NUM_GPUS > 1:
    print(f"Using DataParallel with {NUM_GPUS} GPUs")
    model = nn.DataParallel(model)
    model = model.cuda()
else:
    model = model.cuda()

print(f"✓ Model ready on {NUM_GPUS} GPU(s)")

# Count parameters
if isinstance(model, nn.DataParallel):
    total_params = sum(p.numel() for p in model.module.parameters())
else:
    total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")

HGRNBitForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Initializing model for training...
✓ Gradient checkpointing enabled
Using DataParallel with 4 GPUs
✓ Model ready on 4 GPU(s)
✓ Total parameters: 410,972,160 (411.0M)


In [9]:
# Cell 9: Setup Optimizer and Scheduler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

# Scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=MAX_STEPS,
    eta_min=1e-5
)

print("✓ Optimizer: AdamW")
print("✓ Scheduler: CosineAnnealingLR")
print(f"  Initial LR: {LEARNING_RATE}")
print(f"  Final LR: 1e-5")

✓ Optimizer: AdamW
✓ Scheduler: CosineAnnealingLR
  Initial LR: 0.0003
  Final LR: 1e-5


In [10]:
# Cell 10: Setup DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE_PER_GPU,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

print(f"✓ DataLoader ready")
print(f"  Batch size: {BATCH_SIZE_PER_GPU}")
print(f"  Num workers: 4")
print(f"  Batches per epoch: {len(dataloader):,}")

✓ DataLoader ready
  Batch size: 4
  Num workers: 4
  Batches per epoch: 196,521


## Training Loop

In [ ]:
# Cell 11: Training Loop
import time

print("="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Model: MatMul-Free LM (~{total_params/1e6:.0f}M params)")
print(f"Target: {MAX_STEPS:,} steps")
print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE}")
print(f"Sequence length: {dataset.seq_length}")
print(f"Start time: {datetime.now()}")
print("="*70)

model.train()
global_step = 0
losses = []

data_iter = iter(dataloader)
start_time = time.time()

while global_step < MAX_STEPS:
    try:
        input_ids = next(data_iter)
    except StopIteration:
        data_iter = iter(dataloader)
        input_ids = next(data_iter)
    
    input_ids = input_ids.cuda()
    
    # Forward pass
    outputs = model(input_ids, labels=input_ids)
    loss = outputs.loss
    
    # Handle DataParallel loss gathering
    if isinstance(loss, torch.Tensor) and loss.dim() > 0:
        loss = loss.mean()
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    
    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    
    # Optimizer step
    optimizer.step()
    scheduler.step()
    
    # Logging
    losses.append(loss.item())
    global_step += 1
    
    if global_step % LOG_INTERVAL == 0:
        avg_loss = np.mean(losses[-LOG_INTERVAL:])
        lr = scheduler.get_last_lr()[0]
        elapsed = time.time() - start_time
        tokens_per_sec = (global_step * EFFECTIVE_BATCH_SIZE * dataset.seq_length) / elapsed
        
        print(f"Step {global_step:5d} | Loss: {avg_loss:.4f} | LR: {lr:.2e} | {tokens_per_sec:.0f} tok/s")
    
    # Save checkpoint
    if global_step % SAVE_INTERVAL == 0:
        checkpoint_path = f"checkpoint_v4_step_{global_step}.pt"
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), checkpoint_path)
        else:
            torch.save(model.state_dict(), checkpoint_path)
        print(f"  → Saved checkpoint: {checkpoint_path}")

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"Final step: {global_step}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Total time: {(time.time() - start_time)/3600:.2f} hours")
print(f"End time: {datetime.now()}")

STARTING TRAINING
Model: MatMul-Free LM (~411M params)
Target: 10,000 steps
Effective batch size: 16
Sequence length: 1024
Start time: 2025-11-14 14:51:40.219652


/home/ec2-user/anaconda3/envs/matmul-env/lib/python3.11/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step   100 | Loss: 7.7235 | LR: 3.00e-04 | 402 tok/s


## Evaluation: MatMul-Free vs Pythia-410M

Compare memory usage, throughput, and perplexity

In [ ]:
# Cell 12: Memory Efficiency Test
print("="*70)
print("MEMORY EFFICIENCY COMPARISON")
print("="*70)

model.eval()

# Unwrap DataParallel for single GPU testing
if isinstance(model, nn.DataParallel):
    test_model = model.module.cuda(0)
else:
    test_model = model.cuda(0)

seq_lengths = [128, 256, 512, 1024, 2048]
matmul_free_memory = {}

print("\nMatMul-Free Model:")
for seq_len in seq_lengths:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device=0)
    
    input_ids = torch.randint(0, 50000, (1, seq_len)).cuda(0)
    
    with torch.no_grad():
        outputs = test_model(input_ids)
    
    peak_mem = torch.cuda.max_memory_allocated(device=0) / 1e9
    matmul_free_memory[seq_len] = peak_mem
    print(f"  Seq {seq_len:4d}: {peak_mem:.3f} GB")

print("\n✓ Memory profiling complete")

In [ ]:
# Cell 13: Load Pythia-410M Baseline
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading Pythia-410M baseline...")

baseline_model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/pythia-410m-deduped",
    torch_dtype=torch.float16,
    device_map="cuda:1",  # Use GPU 1
    trust_remote_code=True
)
baseline_tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-410m-deduped")

baseline_params = sum(p.numel() for p in baseline_model.parameters())
print(f"✓ Baseline loaded: {baseline_params:,} parameters ({baseline_params/1e6:.1f}M)")
print(f"  MatMul-Free: {total_params/1e6:.1f}M")
print(f"  Ratio: {total_params/baseline_params:.2f}x")

In [ ]:
# Cell 14: Baseline Memory Test
baseline_model.eval()
baseline_memory = {}

print("\nBaseline Model (Pythia-410M):")
for seq_len in seq_lengths:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device=1)
    
    input_ids = torch.randint(0, 50000, (1, seq_len)).cuda(1)
    
    with torch.no_grad():
        outputs = baseline_model(input_ids)
    
    peak_mem = torch.cuda.max_memory_allocated(device=1) / 1e9
    baseline_memory[seq_len] = peak_mem
    print(f"  Seq {seq_len:4d}: {peak_mem:.3f} GB")

print("\n" + "="*70)
print("MEMORY COMPARISON")
print("="*70)
for seq_len in seq_lengths:
    mf = matmul_free_memory[seq_len]
    bl = baseline_memory[seq_len]
    diff = ((mf - bl) / bl) * 100
    print(f"Seq {seq_len:4d}: MatMul-Free={mf:.3f}GB | Baseline={bl:.3f}GB | Diff={diff:+.1f}%")

In [ ]:
# Cell 15: Throughput Comparison
print("\n" + "="*70)
print("THROUGHPUT COMPARISON")
print("="*70)

batch_size = 4
seq_len = 1024
num_iters = 20

# MatMul-Free throughput
print("\nMatMul-Free Model:")
input_ids = torch.randint(0, 50000, (batch_size, seq_len)).cuda(0)

# Warmup
for _ in range(3):
    with torch.no_grad():
        outputs = test_model(input_ids)

torch.cuda.synchronize(0)
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
with torch.no_grad():
    for _ in range(num_iters):
        outputs = test_model(input_ids)
end.record()

torch.cuda.synchronize(0)
elapsed = start.elapsed_time(end) / 1000.0
mf_throughput = (batch_size * seq_len * num_iters) / elapsed
print(f"  Throughput: {int(mf_throughput):,} tokens/sec")

# Baseline throughput
print("\nBaseline Model (Pythia-410M):")
input_ids = torch.randint(0, 50000, (batch_size, seq_len)).cuda(1)

# Warmup
for _ in range(3):
    with torch.no_grad():
        outputs = baseline_model(input_ids)

torch.cuda.synchronize(1)
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
with torch.no_grad():
    for _ in range(num_iters):
        outputs = baseline_model(input_ids)
end.record()

torch.cuda.synchronize(1)
elapsed = start.elapsed_time(end) / 1000.0
bl_throughput = (batch_size * seq_len * num_iters) / elapsed
print(f"  Throughput: {int(bl_throughput):,} tokens/sec")

speedup = ((mf_throughput - bl_throughput) / bl_throughput) * 100
print(f"\nSpeedup: {speedup:+.1f}%")

# Save results
results_v4 = {
    'matmul_free': {
        'params': total_params,
        'memory': matmul_free_memory,
        'throughput': mf_throughput
    },
    'baseline': {
        'params': baseline_params,
        'memory': baseline_memory,
        'throughput': bl_throughput
    },
    'config': {
        'batch_size': batch_size,
        'seq_len': seq_len,
        'num_iters': num_iters
    }
}

with open('comparison_results_v4.json', 'w') as f:
    json.dump(results_v4, f, indent=2)

print("\n✓ Results saved to comparison_results_v4.json")

In [ ]:
# Cell 16: WikiText-2 Perplexity Evaluation
from datasets import load_dataset
from transformers import AutoTokenizer

print("="*70)
print("WIKITEXT-2 PERPLEXITY EVALUATION")
print("="*70)

# Load WikiText-2
dataset_wikitext = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
tokenizer = AutoTokenizer.from_pretrained("ridger/MMfreeLM-370M")
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

def evaluate_perplexity(model, tokenizer, dataset, device=0, max_samples=None):
    model.eval()
    total_loss = 0
    total_tokens = 0
    num_samples = 0
    
    with torch.no_grad():
        for i, example in enumerate(dataset):
            if max_samples and num_samples >= max_samples:
                break
            
            text = example['text']
            if len(text.strip()) == 0:
                continue
            
            tokens = tokenizer(text, return_tensors='pt', max_length=2048,
                             truncation=True, padding=False)
            input_ids = tokens['input_ids'].cuda(device)
            
            if input_ids.size(1) < 2:
                continue
            
            outputs = model(input_ids, labels=input_ids)
            loss = outputs.loss
            
            total_loss += loss.item() * input_ids.size(1)
            total_tokens += input_ids.size(1)
            num_samples += 1
            
            if (num_samples) % 50 == 0:
                current_ppl = torch.exp(torch.tensor(total_loss / total_tokens))
                print(f"  Processed {num_samples} samples | Current PPL: {current_ppl:.2f}")
    
    final_perplexity = torch.exp(torch.tensor(total_loss / total_tokens))
    return final_perplexity.item(), num_samples, total_tokens

# Evaluate MatMul-Free
print("\nEvaluating MatMul-Free model...")
mf_ppl, mf_samples, mf_tokens = evaluate_perplexity(test_model, tokenizer, dataset_wikitext, device=0)
print(f"✓ MatMul-Free PPL: {mf_ppl:.2f} ({mf_samples} samples, {mf_tokens:,} tokens)")

# Evaluate Baseline
print("\nEvaluating Baseline model...")
bl_ppl, bl_samples, bl_tokens = evaluate_perplexity(baseline_model, baseline_tokenizer, dataset_wikitext, device=1)
print(f"✓ Baseline PPL: {bl_ppl:.2f} ({bl_samples} samples, {bl_tokens:,} tokens)")

print("\n" + "="*70)
print(f"MatMul-Free: {mf_ppl:.2f}")
print(f"Baseline:    {bl_ppl:.2f}")
print(f"Difference:  {mf_ppl - bl_ppl:+.2f}")
print("="*70)

## Summary and Next Steps

In [ ]:
# Cell 17: Final Summary
print("="*70)
print("V4 EXPERIMENT SUMMARY")
print("="*70)
print(f"\nModel Configuration:")
print(f"  MatMul-Free: {total_params/1e6:.0f}M params (24 layers, dim=1024)")
print(f"  Baseline:    {baseline_params/1e6:.0f}M params (Pythia-410M)")

print(f"\nTraining Configuration:")
print(f"  Sequence length: 1024 (vs paper: 1024) ✓")
print(f"  Batch size: {EFFECTIVE_BATCH_SIZE} (vs paper: 256)")
print(f"  Steps: {global_step:,}")
print(f"  Total tokens: {global_step * EFFECTIVE_BATCH_SIZE * dataset.seq_length / 1e9:.2f}B")

print(f"\nPerformance Results:")
print(f"  Memory (seq=2048):  MatMul-Free={matmul_free_memory[2048]:.2f}GB | Baseline={baseline_memory[2048]:.2f}GB")
print(f"  Throughput:         MatMul-Free={int(mf_throughput):,} tok/s | Baseline={int(bl_throughput):,} tok/s")
print(f"  Perplexity:         MatMul-Free={mf_ppl:.2f} | Baseline={bl_ppl:.2f}")

print("\nNext Steps:")
print("  1. If memory/throughput still worse → re-enable autotuning")
print("  2. Train longer (paper used 15B tokens for 370M model)")
print("  3. Try larger batch size if possible")
print("  4. Switch to DDP for better multi-GPU efficiency")
print("="*70)